# SatQuery AI — GeoChat QLoRA fine-tune on RSVQA-LR (Kaggle)

**Before you run — two settings (right sidebar → Notebook options):**
1. **Accelerator = GPU T4 x2** (or P100).
2. **Internet = ON** (needs phone-verified Kaggle account) — required for pip/git/HF downloads.

**How to run it safely (recommended): Save Version → "Save & Run All (Commit)".**
That executes the whole notebook on Kaggle's servers in the background — it keeps
running even if you close the tab or your laptop sleeps, and the trained adapter is
saved to the version's **Output**. (You can also just Run All interactively; a ~45-min
run is short, but Commit is the disconnect-proof option.)

Why the venv: Kaggle ships a newer Python/transformers; GeoChat needs
`transformers==4.31.0` on **Python 3.11**, so we build an isolated 3.11 env.
Big scratch (venv, model, data) lives in `/kaggle/temp`; only the small **adapter**
goes to `/kaggle/working` so it lands in Output.

In [ ]:
#@title 1) Diagnostic — Python, GPU, and internet
import sys, shutil, subprocess
print("kernel python:", sys.version.split()[0])
print("python3.11 present:", bool(shutil.which("python3.11")))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
import urllib.request
try:
    urllib.request.urlopen("https://pypi.org", timeout=8); print("internet: ON")
except Exception as e:
    print("internet: OFF ->", e, "\n>>> Turn Internet ON in Notebook options and re-run.")


In [ ]:
#@title 2) Build the Python-3.11 env with GeoChat's EXACT training stack (~5-8 min)
# Use system python3.11 if present; otherwise install it (with deadsnakes fallback).
!bash -c 'python3.11 --version || (apt-get update -qq && apt-get install -y -qq python3.11 python3.11-venv python3.11-dev) || (apt-get install -y -qq software-properties-common && add-apt-repository -y ppa:deadsnakes/ppa && apt-get update -qq && apt-get install -y -qq python3.11 python3.11-venv python3.11-dev)'
!python3.11 -m venv /kaggle/temp/ft
PY = "/kaggle/temp/ft/bin/python"
!{PY} -m pip install -q --upgrade pip setuptools wheel
!{PY} -m pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
!{PY} -m pip install -q \
  "numpy<2" \
  transformers==4.31.0 peft==0.4.0 accelerate==0.21.0 bitsandbytes==0.41.0 \
  tokenizers==0.13.3 huggingface-hub==0.16.4 sentencepiece==0.1.99 \
  timm==0.6.13 einops==0.6.1 einops-exts==0.0.4 scikit-learn==1.2.2 \
  scipy pillow tifffile protobuf==3.20.3 \
  nvidia-cuda-runtime-cu11 nvidia-cublas-cu11 nvidia-cusparse-cu11
print("install done -> run cell 2b")


In [ ]:
#@title 2b) Put cu11 libs on the loader path (bitsandbytes needs libcusparse), then verify
import glob, os
SP   = "/kaggle/temp/ft/lib/python3.11/site-packages"
libs = sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("LD_LIBRARY_PATH set:\n  " + "\n  ".join(libs))
!/kaggle/temp/ft/bin/python -c "import numpy,torch,sklearn,transformers,peft; \
print('numpy',numpy.__version__,'| torch',torch.__version__,'| cuda',torch.cuda.is_available()); \
import bitsandbytes.functional as F; x=torch.randn(64,64).cuda(); F.quantize_4bit(x); \
print('bitsandbytes 4-bit CUDA OK; all imports OK')"


In [ ]:
#@title 3) Clone PUBLIC GeoChat + editable install into the 3.11 env
!git clone -q https://github.com/mbzuai-oryx/GeoChat.git /kaggle/temp/GeoChat || echo "already cloned"
!/kaggle/temp/ft/bin/python -m pip install -q -e /kaggle/temp/GeoChat --no-deps
!/kaggle/temp/ft/bin/python -c "import geochat; print('geochat import OK')"


In [ ]:
#@title 4) Download GeoChat-7B (~14 GB) to scratch (kernel python is fine here)
from huggingface_hub import snapshot_download
p = snapshot_download("MBZUAI/geochat-7B", local_dir="/kaggle/temp/geochat-7B")
import os; print("downloaded:", p, "->", sorted(os.listdir(p))[:6], "...")


In [ ]:
#@title 5) Download RSVQA-LR (772 tiles, 77k Q/A) — verified Zenodo URLs
import os
os.makedirs("/kaggle/temp/rsvqa", exist_ok=True)
os.chdir("/kaggle/temp/rsvqa")
!wget -q "https://zenodo.org/records/6344334/files/Images_LR.zip?download=1"                -O Images_LR.zip
!wget -q "https://zenodo.org/records/6344334/files/LR_split_train_questions.json?download=1" -O train_questions.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_train_answers.json?download=1"   -O train_answers.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_test_questions.json?download=1"  -O test_questions.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_test_answers.json?download=1"    -O test_answers.json
!unzip -q -o Images_LR.zip
!find . -name "*.tif" | head -3
os.chdir("/kaggle/working")


In [ ]:
%%writefile /kaggle/temp/build_data.py
import json, os, random, glob
from PIL import Image
import numpy as np

random.seed(0)
ROOT   = "/kaggle/temp/rsvqa"
IMGOUT = "/kaggle/temp/rsvqa/images_jpg"
os.makedirs(IMGOUT, exist_ok=True)

tifs = glob.glob(os.path.join(ROOT, "**", "*.tif"), recursive=True)
byid = {}
for t in tifs:
    b = os.path.splitext(os.path.basename(t))[0]
    if b.isdigit():
        byid[int(b)] = t
print("found", len(byid), "tif images")

def to_jpg(i):
    out = os.path.join(IMGOUT, f"{i}.jpg")
    if os.path.exists(out): return out
    src = byid.get(i)
    if src is None: return None
    try:
        im = Image.open(src).convert("RGB")
    except Exception:
        import tifffile
        a = tifffile.imread(src).astype("float32")
        if a.ndim == 2: a = np.stack([a]*3, -1)
        a = a[..., :3] if a.shape[-1] >= 3 else np.repeat(a[..., :1], 3, -1)
        ch = []
        for k in range(3):
            c = a[..., k]; lo, hi = np.percentile(c, 2), np.percentile(c, 98)
            hi = hi if hi > lo else lo + 1
            ch.append(np.clip((c - lo) / (hi - lo), 0, 1))
        im = Image.fromarray((np.stack(ch, -1) * 255).astype("uint8"), "RGB")
    im.save(out, quality=92); return out

def load_split(qf, af):
    Q = json.load(open(os.path.join(ROOT, qf)))["questions"]
    A = json.load(open(os.path.join(ROOT, af)))["answers"]
    print(f"\n[{qf}] sample question:", Q[0])
    print(f"[{af}] sample answer  :", A[0])
    ans_by_id  = {a["id"]: a["answer"] for a in A if "answer" in a}
    ans_by_qid = {a["question_id"]: a["answer"] for a in A if "question_id" in a and "answer" in a}
    rows = []
    for q in Q:
        if q.get("active") is False: continue
        ans = None
        if q.get("answers_ids") and q["answers_ids"][0] in ans_by_id:
            ans = ans_by_id[q["answers_ids"][0]]
        elif q["id"] in ans_by_qid:
            ans = ans_by_qid[q["id"]]
        elif q["id"] in ans_by_id:
            ans = ans_by_id[q["id"]]
        if ans is None: continue
        rows.append({"img_id": q["img_id"], "question": q["question"],
                     "answer": str(ans), "type": q.get("type", "")})
    print(f"  -> resolved {len(rows)} / {len(Q)} questions")
    return rows

SUF = "\nAnswer the question using a single word or short phrase."
def to_llava(rows, n, split):
    random.shuffle(rows); rows = rows[:n]; out = []
    for i, r in enumerate(rows):
        jp = to_jpg(r["img_id"])
        if jp is None: continue
        out.append({"id": f"{split}_{i}", "image": os.path.basename(jp),
                    "conversations": [
                        {"from": "human", "value": "<image>\n" + r["question"] + SUF},
                        {"from": "gpt",   "value": r["answer"]}],
                    "type": r["type"]})
    return out

N_TRAIN = int(os.environ.get("N_TRAIN", "500"))
N_TEST  = int(os.environ.get("N_TEST",  "500"))
train = to_llava(load_split("train_questions.json", "train_answers.json"), N_TRAIN, "train")
test  = to_llava(load_split("test_questions.json",  "test_answers.json"),  N_TEST,  "test")
json.dump(train, open("/kaggle/temp/rsvqa_train_llava.json", "w"))
json.dump(test,  open("/kaggle/temp/rsvqa_test_llava.json",  "w"))
print("\nWROTE train", len(train), "test", len(test), "| jpgs:", len(os.listdir(IMGOUT)))
print("example:", json.dumps(train[0], indent=2)[:400])


In [ ]:
#@title 6b) Run the converter (set N_TRAIN / N_TEST here)
!N_TRAIN=500 N_TEST=500 /kaggle/temp/ft/bin/python /kaggle/temp/build_data.py

In [ ]:
%%writefile /kaggle/temp/eval_rsvqa.py
import json, os, sys, torch
from PIL import Image
from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token, KeywordsStoppingCriteria
from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from geochat.conversation import conv_templates, SeparatorStyle

MODEL   = "/kaggle/temp/geochat-7B"
IMG_DIR = "/kaggle/temp/rsvqa/images_jpg"
TEST    = json.load(open("/kaggle/temp/rsvqa_test_llava.json"))
adapter = sys.argv[1] if len(sys.argv) > 1 else None
LIMIT   = int(os.environ.get("N_EVAL", str(len(TEST))))

name = get_model_name_from_path(MODEL)
tok, model, image_processor, _ = load_pretrained_model(MODEL, None, name, load_4bit=True, device="cuda")
if adapter:
    from peft import PeftModel
    print("attaching adapter:", adapter)
    model = PeftModel.from_pretrained(model, adapter)
model.eval()

def ask(img, q):
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + q)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()
    ids = tokenizer_image_token(prompt, tok, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).cuda()
    it = process_images([img], image_processor, model.config)
    it = [t.to("cuda", dtype=torch.float16) for t in it] if isinstance(it, list) else it.to("cuda", dtype=torch.float16)
    stop = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    sc = KeywordsStoppingCriteria([stop], tok, ids)
    with torch.inference_mode():
        out = model.generate(input_ids=ids, images=it, do_sample=False, max_new_tokens=16,
                             use_cache=True, stopping_criteria=[sc])
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).replace(stop, "").strip()

norm = lambda s: s.strip().lower().rstrip(".")
c = t = 0; per = {}
for r in TEST[:LIMIT]:
    q  = r["conversations"][0]["value"].replace("<image>", "").strip()
    gt = norm(r["conversations"][1]["value"])
    img = Image.open(os.path.join(IMG_DIR, r["image"])).convert("RGB")
    pred = norm(ask(img, q))
    ok = pred == gt or gt in pred
    ty = r.get("type", "all"); per.setdefault(ty, [0, 0]); per[ty][1] += 1; per[ty][0] += ok
    c += ok; t += 1
    if t <= 6: print(("OK " if ok else " X ") + f"gt={gt!r:22} pred={pred!r}")
print(f"\n=== {'ADAPTER' if adapter else 'BASE'} accuracy: {c}/{t} = {100*c/max(t,1):.1f}% ===")
for ty, (cc, nn) in per.items(): print(f"  {ty:14s} {100*cc/nn:5.1f}%  ({cc}/{nn})")


In [ ]:
#@title 8) Eval BEFORE fine-tuning (baseline). Lower N_EVAL for a quicker read.
!N_EVAL=500 /kaggle/temp/ft/bin/python /kaggle/temp/eval_rsvqa.py 2>&1 | tail -25

In [ ]:
#@title 9) TRAIN the LoRA adapter (QLoRA 4-bit). ~45-60 min for N_TRAIN=500.
# Adapter -> /kaggle/working (saved to Output). Checkpoints every 10 steps.
# The '|| echo' keeps the notebook running if GeoChat's final save hits the known
# deepspeed bug — cell 9b recovers the adapter from the last checkpoint.
# Pin to ONE GPU: GeoChat's train script breaks under multi-GPU DataParallel (T4 x2).
!CUDA_VISIBLE_DEVICES=0 /kaggle/temp/ft/bin/python /kaggle/temp/GeoChat/geochat/train/train.py \
  --lora_enable True --lora_r 64 --lora_alpha 16 --lora_dropout 0.05 \
  --bits 4 \
  --model_name_or_path /kaggle/temp/geochat-7B \
  --version v1 \
  --data_path /kaggle/temp/rsvqa_train_llava.json \
  --image_folder /kaggle/temp/rsvqa/images_jpg \
  --vision_tower openai/clip-vit-large-patch14-336 \
  --mm_projector_type mlp2x_gelu \
  --mm_vision_select_layer -2 \
  --mm_use_im_start_end False \
  --mm_use_im_patch_token False \
  --image_aspect_ratio pad \
  --fp16 True --bf16 False --tf32 False \
  --output_dir /kaggle/working/geochat-rsvqa-lora \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --evaluation_strategy no \
  --save_strategy steps --save_steps 10 --save_total_limit 3 \
  --learning_rate 2e-4 --weight_decay 0. --warmup_ratio 0.03 --lr_scheduler_type cosine \
  --logging_steps 5 \
  --model_max_length 2048 \
  --gradient_checkpointing True \
  --lazy_preprocess True \
  --dataloader_num_workers 2 \
  --report_to none \
 || echo "NOTE: final save crashed (deepspeed) — harmless; cell 9b recovers from checkpoint"


In [ ]:
#@title 9b) Recover the adapter from the latest checkpoint (defeats the deepspeed save-crash)
import os, glob, shutil
d = "/kaggle/working/geochat-rsvqa-lora"
top = [f for f in os.listdir(d) if f.startswith("adapter")] if os.path.exists(d) else []
if not top:  # final save didn't write adapter -> promote the last checkpoint
    cks = sorted(glob.glob(d + "/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
    assert cks, "no checkpoints found — training may not have run"
    last = cks[-1]; print("promoting adapter from", last)
    for f in ["adapter_config.json", "adapter_model.bin", "adapter_model.safetensors"]:
        s = os.path.join(last, f)
        if os.path.exists(s): shutil.copy(s, os.path.join(d, f)); print("  copied", f)
print("adapter files:", [f for f in os.listdir(d) if f.startswith("adapter")])


In [ ]:
#@title 10) Eval AFTER fine-tuning (use the SAME N_EVAL as cell 8) — your before/after number
!N_EVAL=500 /kaggle/temp/ft/bin/python /kaggle/temp/eval_rsvqa.py /kaggle/working/geochat-rsvqa-lora 2>&1 | tail -25

In [ ]:
#@title 11) Confirm the adapter is in Output (auto-saved on Commit; else Data->Output to download)
import os
d = "/kaggle/working/geochat-rsvqa-lora"
print("saved to Output:", d)
for f in sorted(os.listdir(d)):
    print("  ", f, os.path.getsize(os.path.join(d, f)) // 1024, "KB")
# optional: a single zip is easier to download from the Output tab
import shutil; shutil.make_archive("/kaggle/working/geochat-rsvqa-lora", "zip", d)
print("also zipped -> /kaggle/working/geochat-rsvqa-lora.zip")


## After it finishes

**Where's the adapter?** In this notebook version's **Output** (`/kaggle/working/geochat-rsvqa-lora`
+ a `.zip`). On a Commit run it's saved automatically; download it from the **Output** tab.

**Your deliverable evidence:** cell 8 (BASE %) vs cell 10 (ADAPTER %) on the same test set —
the mandatory "remote-sensing adaptation" result.

**Wire it into the live demo** (backend already supports it — no code change):
1. Unzip into `D:\SIH2026\models\geochat-rsvqa-lora`.
2. Set `LORA_ADAPTER` to that path (config.py reads it) and start `serve.py`.
3. Startup log shows `attaching LoRA adapter: ...` → the website now runs your fine-tuned GeoChat.

**To push accuracy:** raise `N_TRAIN` (cell 6b) and/or `num_train_epochs` (cell 9).